# 37 - QUAL-004b: evaluación oficial con mapeo de labels corregido

Este notebook corre **QUAL-004b** en Colab usando el evaluador oficial `scripts/evaluate_model.py` del repo `PFI_MVPTest_Enzo_AImodule`, después del ticket **QUAL-003b**.

Flujo:
1. Montar Drive e instalar dependencias.
2. Clonar el AI Module actualizado.
3. Verificar que `evaluate_model.py --help` tenga `--label-map`.
4. Armar un held-out local preliminar con pares imagen/máscara. Para sagital, convierte imagen y máscara MHA a pares 2D .npy alineados, porque el evaluador oficial soporta .mha para imágenes pero no para máscaras.
5. Crear `axial_label_map_alkafri.json`.
6. Ejecutar:
   - `qual-004b-sagittal.json`
   - `qual-004b-axial.json`
7. Generar summary, logs y zip de evidencia.

Mientras `SELECTION_MODE != "clean_test_from_split"`, reportar como **evaluación preliminar con ground truth**, no como test limpio final por paciente.

In [1]:
# ============================================
# 0) Montar Google Drive e instalar dependencias
# ============================================
from google.colab import drive
drive.mount('/content/drive')

!pip -q install SimpleITK pydicom nibabel pandas pillow pytest

Mounted at /content/drive
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.8/52.8 MB 19.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 62.5 MB/s eta 0:00:00


In [2]:
# ============================================
# 1) Configuración principal
# ============================================
from pathlib import Path
import os, sys, json, shutil, subprocess, hashlib, time, zipfile
from datetime import datetime, timezone

import numpy as np
import pandas as pd

DRIVE_ROOT = Path("/content/drive/MyDrive")
PFI_ROOT = DRIVE_ROOT / "PFI_MVP"

REPO_URL = "https://github.com/EnzoAA004/PFI_MVPTest_Enzo_AImodule.git"
REPO_ROOT = Path("/content/PFI_MVPTest_Enzo_AImodule")
REPO_REF = None  # cambiar si querés una branch/commit específico

SAGITTAL_CKPT = PFI_ROOT / "models/final/sagittal_spider_multiclass_final_best.pt"
AXIAL_CKPT    = PFI_ROOT / "models/final/axial_t2_alkafri_final_best.pt"

LOCAL_WORK_ROOT = Path("/content/pfi_qual004b_work")
LOCAL_HELDOUT_ROOT = LOCAL_WORK_ROOT / "heldout"
LOCAL_REPORTS_ROOT = LOCAL_WORK_ROOT / "reports"
LOCAL_DOCS_ROOT = LOCAL_WORK_ROOT / "docs"

SAG_TEST_DIR = LOCAL_HELDOUT_ROOT / "sagittal"
AX_TEST_DIR  = LOCAL_HELDOUT_ROOT / "axial"

SAG_IMAGES_OUT = SAG_TEST_DIR / "images"
SAG_MASKS_OUT  = SAG_TEST_DIR / "masks"
AX_IMAGES_OUT  = AX_TEST_DIR / "images"
AX_MASKS_OUT   = AX_TEST_DIR / "masks"

DRIVE_QUAL_ROOT = PFI_ROOT / "qual"
DRIVE_REPORTS_ROOT = DRIVE_QUAL_ROOT / "reports"
DRIVE_DOCS_ROOT = PFI_ROOT / "docs"

SAG_OUT_JSON = DRIVE_DOCS_ROOT / "qual-004b-sagittal.json"
AX_OUT_JSON  = DRIVE_DOCS_ROOT / "qual-004b-axial.json"
SAG_LOG_PATH = DRIVE_DOCS_ROOT / "qual-004b-sagittal.log"
AX_LOG_PATH  = DRIVE_DOCS_ROOT / "qual-004b-axial.log"
SUMMARY_PATH = DRIVE_DOCS_ROOT / "qual-004b-summary.md"

AX_LABEL_MAP_PATH = DRIVE_REPORTS_ROOT / "axial_label_map_alkafri.json"
MANIFEST_JSON = DRIVE_REPORTS_ROOT / "qual-004b-heldout-manifest.json"
MANIFEST_CSV  = DRIVE_REPORTS_ROOT / "qual-004b-heldout-manifest.csv"
EVIDENCE_ZIP  = DRIVE_REPORTS_ROOT / "qual-004b-evidence.zip"

SELECTION_MODE = "preliminary_gt_not_confirmed_unseen"
CLEAR_LOCAL_HELDOUT = True
TARGET_SIZE = 256

for d in [
    LOCAL_WORK_ROOT, LOCAL_REPORTS_ROOT, LOCAL_DOCS_ROOT,
    SAG_IMAGES_OUT, SAG_MASKS_OUT, AX_IMAGES_OUT, AX_MASKS_OUT,
    DRIVE_REPORTS_ROOT, DRIVE_DOCS_ROOT
]:
    d.mkdir(parents=True, exist_ok=True)

print("PFI_ROOT:", PFI_ROOT, "| exists:", PFI_ROOT.exists())
print("SAGITTAL_CKPT:", SAGITTAL_CKPT, "| exists:", SAGITTAL_CKPT.exists())
print("AXIAL_CKPT:", AXIAL_CKPT, "| exists:", AXIAL_CKPT.exists())
print("REPO_ROOT:", REPO_ROOT)
print("LOCAL_WORK_ROOT:", LOCAL_WORK_ROOT)
print("SELECTION_MODE:", SELECTION_MODE)

PFI_ROOT: /content/drive/MyDrive/PFI_MVP | exists: True
SAGITTAL_CKPT: /content/drive/MyDrive/PFI_MVP/models/final/sagittal_spider_multiclass_final_best.pt | exists: True
AXIAL_CKPT: /content/drive/MyDrive/PFI_MVP/models/final/axial_t2_alkafri_final_best.pt | exists: True
REPO_ROOT: /content/PFI_MVPTest_Enzo_AImodule
LOCAL_WORK_ROOT: /content/pfi_qual004b_work
SELECTION_MODE: preliminary_gt_not_confirmed_unseen


In [3]:
# ============================================
# 2) Utilidades comunes
# ============================================
def sha256_file(path: Path, chunk_size: int = 1024 * 1024) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            b = f.read(chunk_size)
            if not b:
                break
            h.update(b)
    return h.hexdigest()


def reset_dir(path: Path):
    if path.exists() and CLEAR_LOCAL_HELDOUT:
        shutil.rmtree(path)
    path.mkdir(parents=True, exist_ok=True)


def remount_drive_if_needed():
    try:
        from google.colab import drive as colab_drive
        print("Intentando force_remount de Google Drive...")
        colab_drive.mount('/content/drive', force_remount=True)
    except Exception as e:
        print("No pude remontar automáticamente:", repr(e))


def copy_file_robust(src: Path, dst: Path, retries: int = 3, sleep_s: float = 2.0):
    src = Path(src)
    dst = Path(dst)
    dst.parent.mkdir(parents=True, exist_ok=True)
    last_error = None

    for attempt in range(1, retries + 1):
        try:
            shutil.copyfile(src, dst)
            return dst
        except OSError as e:
            last_error = e
            print(f"Copy failed attempt {attempt}/{retries}: {src} -> {dst} | {repr(e)}")
            if getattr(e, "errno", None) == 107 or "Transport endpoint" in str(e):
                remount_drive_if_needed()
            time.sleep(sleep_s)
        except Exception as e:
            last_error = e
            print(f"Copy failed attempt {attempt}/{retries}: {src} -> {dst} | {repr(e)}")
            time.sleep(sleep_s)

    raise RuntimeError(f"No se pudo copiar {src} -> {dst}. Último error: {repr(last_error)}")


def run_cmd(cmd, cwd=None, env=None, log_path=None, check=True):
    print("\n" + "=" * 100)
    print("CMD:", " ".join(str(x) for x in cmd))
    if cwd:
        print("CWD:", cwd)

    proc = subprocess.run(
        [str(x) for x in cmd],
        cwd=str(cwd) if cwd else None,
        env=env,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
    )
    output = proc.stdout or ""
    print(output[-12000:])
    print("returncode:", proc.returncode)

    if log_path:
        Path(log_path).parent.mkdir(parents=True, exist_ok=True)
        Path(log_path).write_text(output, encoding="utf-8", errors="ignore")
        print("log:", log_path)

    if check and proc.returncode != 0:
        raise RuntimeError(f"Comando falló con returncode={proc.returncode}. Ver log={log_path}")

    return proc


def print_tree_counts():
    print("Sagital images:", len(list(SAG_IMAGES_OUT.glob("*"))))
    print("Sagital masks:", len(list(SAG_MASKS_OUT.glob("*"))))
    print("Axial images:", len(list(AX_IMAGES_OUT.glob("*"))))
    print("Axial masks:", len(list(AX_MASKS_OUT.glob("*"))))

In [4]:
# ============================================
# 3) Clonar / actualizar AI Module
# ============================================
%cd /content
!rm -rf PFI_MVPTest_Enzo_AImodule
!git clone {REPO_URL} PFI_MVPTest_Enzo_AImodule

if REPO_REF is not None:
    %cd /content/PFI_MVPTest_Enzo_AImodule
    !git checkout {REPO_REF}

%cd /content/PFI_MVPTest_Enzo_AImodule
!git log --oneline -5
!find /content/PFI_MVPTest_Enzo_AImodule -name "evaluate_model.py" -print
!find /content/PFI_MVPTest_Enzo_AImodule -name "model_architectures.py" -print
!find /content/PFI_MVPTest_Enzo_AImodule -name "test_qual003_metrics.py" -print

/content
Cloning into 'PFI_MVPTest_Enzo_AImodule'...
remote: Enumerating objects: 1489, done.
remote: Counting objects: 100% (322/322), done.
remote: Compressing objects: 100% (235/235), done.
remote: Total 1489 (delta 216), reused 149 (delta 83), pack-reused 1167 (from 2)
Receiving objects: 100% (1489/1489), 13.67 MiB | 26.67 MiB/s, done.
Resolving deltas: 100% (903/903), done.
/content/PFI_MVPTest_Enzo_AImodule
2369030 (HEAD -> main, origin/main, origin/HEAD) Creado con Colab
3a5739b Create 37_qual_004b_official_evaluation_v2_fixed_sagittal_masks.ipynb
c3ac996 qual-003
23ad3e6 Creado con Colab
fd55299 Merge branch 'main' of https://github.com/EnzoAA004/PFI_MVPTest_Enzo_AImodule
/content/PFI_MVPTest_Enzo_AImodule/scripts/evaluate_model.py
/content/PFI_MVPTest_Enzo_AImodule/ai_service/pfi_ai_service/model_architectures.py
/content/PFI_MVPTest_Enzo_AImodule/ai_service/tests/test_qual003_metrics.py


In [5]:
# ============================================
# 4) Verificar evaluador oficial corregido
# ============================================
EVALUATE_SCRIPT = REPO_ROOT / "scripts/evaluate_model.py"
assert EVALUATE_SCRIPT.exists(), f"No existe evaluate_model.py: {EVALUATE_SCRIPT}"

env = os.environ.copy()
env["PYTHONPATH"] = str(REPO_ROOT / "ai_service")

help_proc = run_cmd(
    [sys.executable, str(EVALUATE_SCRIPT), "--help"],
    cwd=REPO_ROOT,
    env=env,
    log_path=LOCAL_DOCS_ROOT / "qual-004b-evaluator-help.log",
    check=True,
)

help_text = help_proc.stdout or ""
if "--label-map" not in help_text:
    raise RuntimeError(
        "El evaluador oficial NO muestra --label-map. "
        "Colab no tiene el commit de QUAL-003b o los cambios no fueron pusheados."
    )

print("OK: evaluate_model.py expone --label-map.")

script_text = EVALUATE_SCRIPT.read_text(encoding="utf-8", errors="ignore")
print("Contiene label_group_mapping:", "label_group_mapping" in script_text)
print("Contiene macroForegroundReliable:", "macroForegroundReliable" in script_text or "macro_foreground_reliable" in script_text)
print("Contiene gt_present_cases:", "gt_present_cases" in script_text)


CMD: /usr/bin/python3 /content/PFI_MVPTest_Enzo_AImodule/scripts/evaluate_model.py --help
CWD: /content/PFI_MVPTest_Enzo_AImodule
usage: evaluate_model.py [-h] --plane {sagittal,axial} --checkpoint CHECKPOINT
                         --test-dir TEST_DIR --num-classes NUM_CLASSES
                         [--target-size TARGET_SIZE] [--label-map LABEL_MAP]
                         [--output OUTPUT]

Evalua Dice/IoU por clase contra un held-out local deidentificado.

options:
  -h, --help            show this help message and exit
  --plane {sagittal,axial}
  --checkpoint CHECKPOINT
  --test-dir TEST_DIR   Directorio con images/ y masks/.
  --num-classes NUM_CLASSES
  --target-size TARGET_SIZE
  --label-map LABEL_MAP
                        JSON opcional raw_label -> class_id o class_id ->
                        [raw_labels].
  --output OUTPUT

returncode: 0
log: /content/pfi_qual004b_work/docs/qual-004b-evaluator-help.log
OK: evaluate_model.py expone --label-map.
Contiene label_group_m

In [6]:
# ============================================
# 5) Opcional: correr tests QUAL-003b en Colab
# ============================================
test_candidates = list(REPO_ROOT.rglob("test_qual003_metrics.py"))
print("test_qual003_metrics.py encontrados:", test_candidates)

if test_candidates:
    run_cmd(
        [sys.executable, "-m", "pytest", str(test_candidates[0]), "-q"],
        cwd=REPO_ROOT,
        env=env,
        log_path=LOCAL_DOCS_ROOT / "qual-003b-tests-colab.log",
        check=True,
    )
else:
    print("No encontré test_qual003_metrics.py. Continúo solo con verificación de --label-map.")

test_qual003_metrics.py encontrados: [PosixPath('/content/PFI_MVPTest_Enzo_AImodule/ai_service/tests/test_qual003_metrics.py')]

CMD: /usr/bin/python3 -m pytest /content/PFI_MVPTest_Enzo_AImodule/ai_service/tests/test_qual003_metrics.py -q
CWD: /content/PFI_MVPTest_Enzo_AImodule
.......                                                                  [100%]
7 passed in 4.26s

returncode: 0
log: /content/pfi_qual004b_work/docs/qual-003b-tests-colab.log


In [7]:
# ============================================
# 6) Buscar splits/inventarios existentes para trazabilidad
# ============================================
patterns = [
    "*split*", "*overview*", "*holdout*", "*test*", "*train*", "*validation*", "*val*",
    "*pairing*", "*inventory*", "*manifest*"
]
valid_suffixes = {".csv", ".json", ".txt", ".parquet", ".tsv"}

hits = []
for pat in patterns:
    for p in PFI_ROOT.rglob(pat):
        if p.is_file() and p.suffix.lower() in valid_suffixes:
            normalized = str(p).replace("\\", "/").lower()
            if "/qual/heldout/" in normalized:
                continue
            hits.append(p)

hits = sorted(set(hits), key=lambda p: str(p).lower())
print("Archivos candidatos de split/inventario:", len(hits))
for i, p in enumerate(hits[:250]):
    print(f"{i:03d} | {p}")

split_scan_path = DRIVE_REPORTS_ROOT / "qual-004b-split-candidates.txt"
split_scan_path.write_text("\\n".join(str(p) for p in hits), encoding="utf-8")
print("\\nGuardado:", split_scan_path)

Archivos candidatos de split/inventario: 89
000 | /content/drive/MyDrive/PFI_MVP/data/SPIDER/overview.csv
001 | /content/drive/MyDrive/PFI_MVP/qual/reports/qual-002-heldout-manifest.csv
002 | /content/drive/MyDrive/PFI_MVP/qual/reports/qual-002-heldout-manifest.json
003 | /content/drive/MyDrive/PFI_MVP/qual/reports/qual-002-split-candidates.txt
004 | /content/drive/MyDrive/PFI_MVP/qual/reports/qual-004b-heldout-manifest.csv
005 | /content/drive/MyDrive/PFI_MVP/qual/reports/qual-004b-heldout-manifest.json
006 | /content/drive/MyDrive/PFI_MVP/qual/reports/qual-004b-split-candidates.txt
007 | /content/drive/MyDrive/PFI_MVP/results/E10_axial_t2_final_training_clean/E10_axial_t2_final_training_clean_report.json
008 | /content/drive/MyDrive/PFI_MVP/results/E10_axial_t2_final_training_clean/E10_test_metrics.json
009 | /content/drive/MyDrive/PFI_MVP/results/E10_axial_t2_final_training_clean/E10_training_history.csv
010 | /content/drive/MyDrive/PFI_MVP/results/E10_axial_t2_final_training_clean/

In [8]:
# ============================================
# 7) Verificar pares sagitales SPIDER
# ============================================
SAG_IMAGES_SRC = PFI_ROOT / "data/SPIDER/images/images"
SAG_MASKS_SRC = PFI_ROOT / "data/SPIDER/masks/masks"

print("SAG_IMAGES_SRC:", SAG_IMAGES_SRC, "| exists:", SAG_IMAGES_SRC.exists())
print("SAG_MASKS_SRC:", SAG_MASKS_SRC, "| exists:", SAG_MASKS_SRC.exists())

def find_sag_mask_for_image(img: Path):
    exact = SAG_MASKS_SRC / img.name
    if exact.exists():
        return exact

    candidates = [p for p in SAG_MASKS_SRC.glob(img.stem + "*") if p.is_file()]
    if candidates:
        return sorted(candidates, key=lambda p: str(p))[0]

    case_id = img.name.split("_")[0]
    candidates = [p for p in SAG_MASKS_SRC.glob(case_id + "*") if p.is_file()]
    if candidates:
        return sorted(candidates, key=lambda p: str(p))[0]

    return None

sag_pairs = []
missing_sag_masks = []

for img in sorted(SAG_IMAGES_SRC.glob("*.mha")):
    if "_SPACE" in img.name:
        continue
    mask = find_sag_mask_for_image(img)
    if mask is not None and mask.exists():
        sag_pairs.append((img, mask))
    else:
        missing_sag_masks.append(img)

print("Pares sagitales encontrados:", len(sag_pairs))
print("Imágenes sagitales sin máscara detectada:", len(missing_sag_masks))
for img, mask in sag_pairs[:20]:
    print(img.name, "|", mask.name)

if not sag_pairs:
    raise RuntimeError("No encontré pares sagitales SPIDER.")

SAG_IMAGES_SRC: /content/drive/MyDrive/PFI_MVP/data/SPIDER/images/images | exists: True
SAG_MASKS_SRC: /content/drive/MyDrive/PFI_MVP/data/SPIDER/masks/masks | exists: True
Pares sagitales encontrados: 406
Imágenes sagitales sin máscara detectada: 0
100_t1.mha | 100_t1.mha
100_t2.mha | 100_t2.mha
101_t1.mha | 101_t1.mha
101_t2.mha | 101_t2.mha
104_t1.mha | 104_t1.mha
104_t2.mha | 104_t2.mha
105_t1.mha | 105_t1.mha
105_t2.mha | 105_t2.mha
106_t1.mha | 106_t1.mha
106_t2.mha | 106_t2.mha
107_t1.mha | 107_t1.mha
107_t2.mha | 107_t2.mha
108_t1.mha | 108_t1.mha
108_t2.mha | 108_t2.mha
109_t1.mha | 109_t1.mha
109_t2.mha | 109_t2.mha
10_t1.mha | 10_t1.mha
10_t2.mha | 10_t2.mha
110_t1.mha | 110_t1.mha
110_t2.mha | 110_t2.mha


In [9]:
# ============================================
# 8) Verificar pares axiales Al-Kafri/Sudirman
# ============================================
AXIAL_PAIRING_SRC = PFI_ROOT / "data/AXIAL_ALKAFRI/processed/pairing_v1"
print("AXIAL_PAIRING_SRC:", AXIAL_PAIRING_SRC, "| exists:", AXIAL_PAIRING_SRC.exists())

axial_pairs = []
for img in sorted(AXIAL_PAIRING_SRC.glob("*_image.npy")):
    mask = AXIAL_PAIRING_SRC / img.name.replace("_image.npy", "_mask.npy")
    if mask.exists():
        pair_id = img.name.replace("_image.npy", "")
        axial_pairs.append((pair_id, img, mask))

print("Pares axiales encontrados:", len(axial_pairs))
for pair_id, img, mask in axial_pairs[:20]:
    print(pair_id, "|", img.name, "|", mask.name)

if axial_pairs:
    arr = np.load(axial_pairs[0][1], allow_pickle=False)
    msk = np.load(axial_pairs[0][2], allow_pickle=False)
    print("\nPrimer axial image shape/dtype/range:", arr.shape, arr.dtype, float(np.nanmin(arr)), float(np.nanmax(arr)))
    print("Primer axial mask shape/dtype/range:", msk.shape, msk.dtype, float(np.nanmin(msk)), float(np.nanmax(msk)))
else:
    raise RuntimeError("No encontré pares axiales pairing_v1.")

AXIAL_PAIRING_SRC: /content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKAFRI/processed/pairing_v1 | exists: True
Pares axiales encontrados: 718
pair_0001 | pair_0001_image.npy | pair_0001_mask.npy
pair_0002 | pair_0002_image.npy | pair_0002_mask.npy
pair_0003 | pair_0003_image.npy | pair_0003_mask.npy
pair_0004 | pair_0004_image.npy | pair_0004_mask.npy
pair_0005 | pair_0005_image.npy | pair_0005_mask.npy
pair_0006 | pair_0006_image.npy | pair_0006_mask.npy
pair_0007 | pair_0007_image.npy | pair_0007_mask.npy
pair_0008 | pair_0008_image.npy | pair_0008_mask.npy
pair_0009 | pair_0009_image.npy | pair_0009_mask.npy
pair_0010 | pair_0010_image.npy | pair_0010_mask.npy
pair_0011 | pair_0011_image.npy | pair_0011_mask.npy
pair_0012 | pair_0012_image.npy | pair_0012_mask.npy
pair_0013 | pair_0013_image.npy | pair_0013_mask.npy
pair_0014 | pair_0014_image.npy | pair_0014_mask.npy
pair_0015 | pair_0015_image.npy | pair_0015_mask.npy
pair_0016 | pair_0016_image.npy | pair_0016_mask.npy
pair_0017 | pair

In [10]:
# ============================================
# 9) Selección preliminar del held-out
# ============================================
USER_SAG_CASE_NAMES = None
USER_AXIAL_PAIR_IDS = None

DEFAULT_SAG_CASE_NAMES = [
    "101_t2.mha", "104_t2.mha", "105_t2.mha", "106_t2.mha", "107_t2.mha",
    "109_t2.mha", "113_t2.mha", "115_t2.mha", "116_t2.mha", "118_t2.mha",
]

DEFAULT_AXIAL_PAIR_IDS = [f"pair_{i:04d}" for i in range(1, 31)]

sag_case_names = USER_SAG_CASE_NAMES or DEFAULT_SAG_CASE_NAMES
axial_pair_ids = USER_AXIAL_PAIR_IDS or DEFAULT_AXIAL_PAIR_IDS

available_sag_by_name = {img.name: (img, mask) for img, mask in sag_pairs}
selected_sag = []
for name in sag_case_names:
    if name in available_sag_by_name:
        selected_sag.append((name, *available_sag_by_name[name]))

if len(selected_sag) < len(sag_case_names):
    already = {name for name, _, _ in selected_sag}
    for img, mask in sag_pairs:
        if img.name not in already:
            selected_sag.append((img.name, img, mask))
        if len(selected_sag) >= len(sag_case_names):
            break

available_ax_by_id = {pair_id: (img, mask) for pair_id, img, mask in axial_pairs}
selected_axial = []
for pair_id in axial_pair_ids:
    if pair_id in available_ax_by_id:
        selected_axial.append((pair_id, *available_ax_by_id[pair_id]))

if len(selected_axial) < len(axial_pair_ids):
    already = {pair_id for pair_id, _, _ in selected_axial}
    for pair_id, img, mask in axial_pairs:
        if pair_id not in already:
            selected_axial.append((pair_id, img, mask))
        if len(selected_axial) >= len(axial_pair_ids):
            break

print("SELECTION_MODE:", SELECTION_MODE)
print("Selected sagittal:", len(selected_sag))
for item in selected_sag:
    print(" -", item[0], "|", item[1], "|", item[2])

print("\nSelected axial:", len(selected_axial))
for item in selected_axial[:40]:
    print(" -", item[0], "|", item[1].name, "|", item[2].name)

if len(selected_sag) == 0:
    raise RuntimeError("No se seleccionó ningún caso sagital.")
if len(selected_axial) == 0:
    raise RuntimeError("No se seleccionó ningún caso axial.")

SELECTION_MODE: preliminary_gt_not_confirmed_unseen
Selected sagittal: 10
 - 101_t2.mha | /content/drive/MyDrive/PFI_MVP/data/SPIDER/images/images/101_t2.mha | /content/drive/MyDrive/PFI_MVP/data/SPIDER/masks/masks/101_t2.mha
 - 104_t2.mha | /content/drive/MyDrive/PFI_MVP/data/SPIDER/images/images/104_t2.mha | /content/drive/MyDrive/PFI_MVP/data/SPIDER/masks/masks/104_t2.mha
 - 105_t2.mha | /content/drive/MyDrive/PFI_MVP/data/SPIDER/images/images/105_t2.mha | /content/drive/MyDrive/PFI_MVP/data/SPIDER/masks/masks/105_t2.mha
 - 106_t2.mha | /content/drive/MyDrive/PFI_MVP/data/SPIDER/images/images/106_t2.mha | /content/drive/MyDrive/PFI_MVP/data/SPIDER/masks/masks/106_t2.mha
 - 107_t2.mha | /content/drive/MyDrive/PFI_MVP/data/SPIDER/images/images/107_t2.mha | /content/drive/MyDrive/PFI_MVP/data/SPIDER/masks/masks/107_t2.mha
 - 109_t2.mha | /content/drive/MyDrive/PFI_MVP/data/SPIDER/images/images/109_t2.mha | /content/drive/MyDrive/PFI_MVP/data/SPIDER/masks/masks/109_t2.mha
 - 113_t2.mha 

In [11]:
# ============================================
# 10) Crear carpetas held-out locales
# ============================================
# IMPORTANTE:
# El evaluador oficial acepta .mha/.mhd como imagen, pero sus máscaras GT soportadas
# son .npy o formatos de imagen 2D. Por eso, para sagital convertimos cada par
# SPIDER .mha volumen -> image_2d.npy + mask_2d.npy, usando el mismo eje guardado
# en el checkpoint sagital. Así el stem coincide y la máscara es leíble por load_mask().

import SimpleITK as sitk
import torch

reset_dir(SAG_IMAGES_OUT)
reset_dir(SAG_MASKS_OUT)
reset_dir(AX_IMAGES_OUT)
reset_dir(AX_MASKS_OUT)

manifest_rows = []

# Resolver eje sagital desde checkpoint, con fallback 2.
sag_ckpt = torch.load(str(SAGITTAL_CKPT), map_location="cpu", weights_only=False)
SAG_AXIS_FOR_2D = int(sag_ckpt.get("sagittal_axis", 2)) if isinstance(sag_ckpt, dict) else 2
print("SAG_AXIS_FOR_2D:", SAG_AXIS_FOR_2D)


def read_mha_array(path: Path) -> np.ndarray:
    img = sitk.ReadImage(str(path))
    return np.asarray(sitk.GetArrayFromImage(img))


def extract_same_slice_2d(image_arr: np.ndarray, mask_arr: np.ndarray, axis: int):
    image_arr = np.squeeze(np.asarray(image_arr))
    mask_arr = np.squeeze(np.asarray(mask_arr))
    if image_arr.ndim == 2:
        if mask_arr.ndim != 2:
            raise ValueError(f"Imagen 2D pero máscara no 2D: image={image_arr.shape}, mask={mask_arr.shape}")
        return image_arr, mask_arr, None
    if image_arr.ndim != 3 or mask_arr.ndim != 3:
        raise ValueError(f"Se esperaban arrays 2D/3D: image={image_arr.shape}, mask={mask_arr.shape}")
    if image_arr.shape != mask_arr.shape:
        raise ValueError(f"Shape mismatch antes de slice: image={image_arr.shape}, mask={mask_arr.shape}")
    axis = int(axis)
    if axis < 0 or axis >= image_arr.ndim:
        axis = int(np.argmin(image_arr.shape))
    slice_index = image_arr.shape[axis] // 2
    image_2d = np.take(image_arr, slice_index, axis=axis)
    mask_2d = np.take(mask_arr, slice_index, axis=axis)
    return image_2d, mask_2d, slice_index


# Sagital: crear pares 2D .npy alineados.
for case_id, img, mask in selected_sag:
    case_stem = Path(case_id).stem
    img_dst = SAG_IMAGES_OUT / f"{case_stem}.npy"
    mask_dst = SAG_MASKS_OUT / f"{case_stem}.npy"

    image_raw = read_mha_array(img)
    mask_raw = read_mha_array(mask)
    image_2d, mask_2d, slice_index = extract_same_slice_2d(image_raw, mask_raw, SAG_AXIS_FOR_2D)

    np.save(img_dst, np.asarray(image_2d, dtype=np.float32))
    np.save(mask_dst, np.asarray(mask_2d, dtype=np.int64))

    unique_mask_values = sorted(int(v) for v in np.unique(mask_2d))[:50]
    print(f"Sagital {case_stem}: src_shape={image_raw.shape}, slice={slice_index}, image2d={image_2d.shape}, mask_unique_sample={unique_mask_values}")

    manifest_rows.append({
        "plane": "sagittal",
        "case_id": case_stem,
        "image_source": str(img),
        "mask_source": str(mask),
        "image_dest_local": str(img_dst),
        "mask_dest_local": str(mask_dst),
        "selection_mode": SELECTION_MODE,
        "sagittal_axis": SAG_AXIS_FOR_2D,
        "slice_index": slice_index,
        "note": "MHA volume converted to aligned 2D NPY pair because official evaluator does not support MHA masks.",
    })

# Axial: ya está en pares .npy 2D; solo renombrar a mismo stem.
for pair_id, img, mask in selected_axial:
    img_dst = AX_IMAGES_OUT / f"{pair_id}.npy"
    mask_dst = AX_MASKS_OUT / f"{pair_id}.npy"
    copy_file_robust(img, img_dst)
    copy_file_robust(mask, mask_dst)

    manifest_rows.append({
        "plane": "axial",
        "case_id": pair_id,
        "image_source": str(img),
        "mask_source": str(mask),
        "image_dest_local": str(img_dst),
        "mask_dest_local": str(mask_dst),
        "selection_mode": SELECTION_MODE,
    })

print_tree_counts()

manifest_df = pd.DataFrame(manifest_rows)
manifest_df.to_csv(MANIFEST_CSV, index=False)
MANIFEST_JSON.write_text(json.dumps({
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "selection_mode": SELECTION_MODE,
    "warning": "Si selection_mode != clean_test_from_split, reportar como evaluación preliminar con ground truth, no como test limpio final.",
    "sagittal_test_dir_local": str(SAG_TEST_DIR),
    "axial_test_dir_local": str(AX_TEST_DIR),
    "sagittal_mha_to_2d_npy": True,
    "sagittal_axis": SAG_AXIS_FOR_2D,
    "counts": {"sagittal": len(selected_sag), "axial": len(selected_axial)},
    "rows": manifest_rows,
}, indent=2, ensure_ascii=False), encoding="utf-8")

print("MANIFEST_CSV:", MANIFEST_CSV)
print("MANIFEST_JSON:", MANIFEST_JSON)
manifest_df.head()


SAG_AXIS_FOR_2D: 2
Sagital 101_t2: src_shape=(352, 384, 17), slice=8, image2d=(352, 384), mask_unique_sample=[0, 1, 2, 3, 4, 5, 6, 100, 201, 202, 203, 204, 205, 206]
Sagital 104_t2: src_shape=(384, 384, 15), slice=7, image2d=(384, 384), mask_unique_sample=[0, 1, 2, 3, 4, 5, 6, 7, 100, 201, 202, 203, 204, 205, 206, 207]
Sagital 105_t2: src_shape=(427, 448, 25), slice=12, image2d=(427, 448), mask_unique_sample=[0, 1, 2, 3, 4, 5, 6, 7, 100, 201, 202, 203, 204, 205, 206, 207]
Sagital 106_t2: src_shape=(384, 384, 15), slice=7, image2d=(384, 384), mask_unique_sample=[0, 1, 2, 3, 4, 5, 6, 7, 100, 201, 202, 203, 204, 205, 206, 207]


RuntimeError: Exception thrown in SimpleITK ImageFileReader_Execute: /work/src/Code/IO/src/sitkImageReaderBase.cxx:91:
sitk::ERROR: The file "/content/drive/MyDrive/PFI_MVP/data/SPIDER/images/images/107_t2.mha" does not exist.

In [ ]:
# ============================================
# 11) Crear axial_label_map_alkafri.json
# ============================================
axial_label_map = {
    "250": 0,
    "0": 1,
    "50": 2,
    "100": 3,
    "150": 4,
    "200": 5
}

AX_LABEL_MAP_PATH.write_text(json.dumps(axial_label_map, indent=2), encoding="utf-8")

print("AX_LABEL_MAP_PATH:", AX_LABEL_MAP_PATH)
print(AX_LABEL_MAP_PATH.read_text())

In [ ]:
# ============================================
# 12) Ejecutar evaluador oficial QUAL-004b
# ============================================
assert SAGITTAL_CKPT.exists(), SAGITTAL_CKPT
assert AXIAL_CKPT.exists(), AXIAL_CKPT
assert (SAG_TEST_DIR / "images").exists(), SAG_TEST_DIR
assert (AX_TEST_DIR / "images").exists(), AX_TEST_DIR
assert AX_LABEL_MAP_PATH.exists(), AX_LABEL_MAP_PATH

env = os.environ.copy()
env["PYTHONPATH"] = str(REPO_ROOT / "ai_service")

print("PYTHONPATH:", env["PYTHONPATH"])
print("Evaluator:", EVALUATE_SCRIPT)

sag_cmd = [
    sys.executable, str(EVALUATE_SCRIPT),
    "--plane", "sagittal",
    "--checkpoint", str(SAGITTAL_CKPT),
    "--test-dir", str(SAG_TEST_DIR),
    "--num-classes", "4",
    "--target-size", str(TARGET_SIZE),
    "--output", str(SAG_OUT_JSON),
]

ax_cmd = [
    sys.executable, str(EVALUATE_SCRIPT),
    "--plane", "axial",
    "--checkpoint", str(AXIAL_CKPT),
    "--test-dir", str(AX_TEST_DIR),
    "--num-classes", "6",
    "--target-size", str(TARGET_SIZE),
    "--label-map", str(AX_LABEL_MAP_PATH),
    "--output", str(AX_OUT_JSON),
]

run_cmd(sag_cmd, cwd=REPO_ROOT, env=env, log_path=SAG_LOG_PATH, check=True)
run_cmd(ax_cmd, cwd=REPO_ROOT, env=env, log_path=AX_LOG_PATH, check=True)

print("Sagital output:", SAG_OUT_JSON, "| exists:", SAG_OUT_JSON.exists())
print("Axial output:", AX_OUT_JSON, "| exists:", AX_OUT_JSON.exists())

In [ ]:
# ============================================
# 13) Leer resultados y generar resumen
# ============================================
def load_json(path: Path):
    if not path.exists():
        print("NO EXISTE:", path)
        return None
    return json.loads(path.read_text(encoding="utf-8"))


def find_key_recursive(obj, keys):
    if obj is None:
        return None
    if isinstance(obj, dict):
        for k, v in obj.items():
            if k in keys:
                return v
        for v in obj.values():
            found = find_key_recursive(v, keys)
            if found is not None:
                return found
    elif isinstance(obj, list):
        for item in obj:
            found = find_key_recursive(item, keys)
            if found is not None:
                return found
    return None


def extract_warnings(obj):
    warnings = find_key_recursive(obj, {"warnings", "Warnings"})
    if warnings is None:
        return []
    if isinstance(warnings, list):
        return warnings
    return [warnings]


def extract_metric(obj, *names):
    return find_key_recursive(obj, set(names))


sag_result = load_json(SAG_OUT_JSON)
ax_result = load_json(AX_OUT_JSON)

print("\n==== qual-004b-sagittal.json ====")
print(json.dumps(sag_result, indent=2, ensure_ascii=False)[:16000] if sag_result else "NO JSON")

print("\n==== qual-004b-axial.json ====")
print(json.dumps(ax_result, indent=2, ensure_ascii=False)[:16000] if ax_result else "NO JSON")

sag_dice_macro = extract_metric(sag_result, "dice_macro_foreground", "diceMacroForeground", "macro_dice_foreground", "dice_macro_no_bg", "diceMacroNoBackground")
sag_iou_macro = extract_metric(sag_result, "iou_macro_foreground", "iouMacroForeground", "macro_iou_foreground", "iou_macro_no_bg", "iouMacroNoBackground")
ax_dice_macro = extract_metric(ax_result, "dice_macro_foreground", "diceMacroForeground", "macro_dice_foreground", "dice_macro_no_bg", "diceMacroNoBackground")
ax_iou_macro = extract_metric(ax_result, "iou_macro_foreground", "iouMacroForeground", "macro_iou_foreground", "iou_macro_no_bg", "iouMacroNoBackground")
ax_dice_excl = extract_metric(ax_result, "dice_macro_excluding_raw0", "diceMacroExcludingRaw0")
ax_iou_excl = extract_metric(ax_result, "iou_macro_excluding_raw0", "iouMacroExcludingRaw0")

sag_reliable = extract_metric(sag_result, "macroForegroundReliable", "macro_foreground_reliable")
ax_reliable = extract_metric(ax_result, "macroForegroundReliable", "macro_foreground_reliable")

sag_warnings = extract_warnings(sag_result)
ax_warnings = extract_warnings(ax_result)

summary_md = f'''# QUAL-004b - Evaluación oficial con mapeo corregido

Fecha UTC: {datetime.now(timezone.utc).isoformat()}

## Advertencia metodológica

selection_mode: `{SELECTION_MODE}`

Si `selection_mode` no es `clean_test_from_split`, estos números deben reportarse como **evaluación preliminar con ground truth**, no como test limpio final por paciente.

## Evaluador

- Repo: `{REPO_ROOT}`
- Script: `{EVALUATE_SCRIPT}`
- `--label-map`: confirmado en help
- PYTHONPATH: `{env["PYTHONPATH"]}`

## Inputs

Sagital:
- test_dir local: `{SAG_TEST_DIR}`
- checkpoint: `{SAGITTAL_CKPT}`
- casos: {len(selected_sag)}

Axial:
- test_dir local: `{AX_TEST_DIR}`
- checkpoint: `{AXIAL_CKPT}`
- label_map: `{AX_LABEL_MAP_PATH}`
- casos: {len(selected_axial)}

## Outputs

- `{SAG_OUT_JSON}`
- `{AX_OUT_JSON}`
- `{SAG_LOG_PATH}`
- `{AX_LOG_PATH}`
- `{MANIFEST_JSON}`

## Métricas principales detectadas

- Sagital Dice macro foreground/no-bg: `{sag_dice_macro}`
- Sagital IoU macro foreground/no-bg: `{sag_iou_macro}`
- Sagital macroForegroundReliable: `{sag_reliable}`

- Axial Dice macro foreground/no-bg: `{ax_dice_macro}`
- Axial IoU macro foreground/no-bg: `{ax_iou_macro}`
- Axial Dice macro excluyendo raw_0: `{ax_dice_excl}`
- Axial IoU macro excluyendo raw_0: `{ax_iou_excl}`
- Axial macroForegroundReliable: `{ax_reliable}`

## Warnings

Sagital:
```text
{json.dumps(sag_warnings, indent=2, ensure_ascii=False)}
```

Axial:
```text
{json.dumps(ax_warnings, indent=2, ensure_ascii=False)}
```

## Nota para Claude/Codex

Estos JSON reemplazan la corrida preliminar del notebook 36 v3 porque ahora usan el evaluador oficial corregido por QUAL-003b.

Usar `qual-004b-sagittal.json` y `qual-004b-axial.json` como evidencia preliminar para decidir macro vs per-clase y planificar QUAL-005, sin presentarlos como test limpio final si no se confirma split limpio por paciente.
'''.strip()

SUMMARY_PATH.write_text(summary_md, encoding="utf-8")
print("\n==== SUMMARY QUAL-004b ====")
print(summary_md)
print("\nGuardado:", SUMMARY_PATH)

In [ ]:
# ============================================
# 14) Comprimir evidencia QUAL-004b
# ============================================
files_to_zip = [
    DRIVE_REPORTS_ROOT / "qual-004b-split-candidates.txt",
    MANIFEST_CSV,
    MANIFEST_JSON,
    AX_LABEL_MAP_PATH,
    SAG_OUT_JSON,
    AX_OUT_JSON,
    SAG_LOG_PATH,
    AX_LOG_PATH,
    SUMMARY_PATH,
    LOCAL_DOCS_ROOT / "qual-004b-evaluator-help.log",
    LOCAL_DOCS_ROOT / "qual-003b-tests-colab.log",
]

with zipfile.ZipFile(EVIDENCE_ZIP, "w", compression=zipfile.ZIP_DEFLATED) as z:
    for p in files_to_zip:
        p = Path(p)
        if p.exists():
            z.write(p, arcname=p.name)
        else:
            print("No existe, no se agrega:", p)

print("EVIDENCE_ZIP:", EVIDENCE_ZIP)
print("exists:", EVIDENCE_ZIP.exists())
if EVIDENCE_ZIP.exists():
    print("size MB:", round(EVIDENCE_ZIP.stat().st_size / (1024 * 1024), 3))
    print("sha256:", sha256_file(EVIDENCE_ZIP))

In [ ]:
# ============================================
# 15) Mensaje sugerido para Claude/Codex
# ============================================
print(f'''
QUAL-004b ejecutado en Colab con evaluador oficial corregido por QUAL-003b.

Held-out:
- selection_mode: {SELECTION_MODE}
- sagital casos: {len(selected_sag)}
- axial casos: {len(selected_axial)}

Evaluador:
- repo: {REPO_ROOT}
- script: {EVALUATE_SCRIPT}
- --label-map confirmado en help

Outputs:
- {SAG_OUT_JSON}
- {AX_OUT_JSON}
- {SUMMARY_PATH}
- {MANIFEST_JSON}
- {EVIDENCE_ZIP}

Label map axial:
- {AX_LABEL_MAP_PATH}
- mapping: 250→0, 0→1, 50→2, 100→3, 150→4, 200→5

Aclaración:
Sigue siendo preliminary_gt_not_confirmed_unseen hasta confirmar split limpio por paciente.

Copio a continuación el summary y/o los JSON.
''')